# 01 — VisDrone preparation (Colab)

Prepare VisDrone for **person detection only** (pedestrian + people → `person`).

This notebook does **not** train models and does **not** create uniform labels
(`our_team` / `known_friendly` / `unknown`).

Edit the **configuration cell** below, then run the rest in order.

## Configuration (edit these)

In [ ]:
# --- EDIT ME ---
REPO_URL = "https://github.com/YOUR_ORG/uniform-detection.git"  # placeholder
PROJECT_DIR = "/content/uniform-detection"
DRIVE_ROOT = "/content/drive/MyDrive/uniform-detection"  # placeholder Drive project path

VISDRONE_RAW_DIR = f"{DRIVE_ROOT}/data/raw/visdrone"
VISDRONE_PROCESSED_DIR = f"{DRIVE_ROOT}/data/processed/visdrone_person"

RUN_CONVERSION = True
OVERWRITE = False
LIMIT = None  # e.g. 50 for a smoke run; None = full split
VIS_COUNT = 8

## 1. Environment information

In [ ]:
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone or update the GitHub repository

In [ ]:
import os
from pathlib import Path

project_path = Path(PROJECT_DIR)
if project_path.exists() and (project_path / ".git").exists():
    %cd {PROJECT_DIR}
    !git pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

print("Project:", Path.cwd())

## 4. Project root and Drive dataset locations

In [ ]:
from pathlib import Path

from src.utils.paths import find_project_root

ROOT = find_project_root()
raw_dir = Path(VISDRONE_RAW_DIR)
processed_dir = Path(VISDRONE_PROCESSED_DIR)

print("Repo root:", ROOT)
print("VisDrone raw:", raw_dir)
print("VisDrone processed:", processed_dir)

## 5. Install project dependencies

In [ ]:
!pip install -q -r requirements.txt

## 6. Verify expected VisDrone dataset location

In [ ]:
from src.data.visdrone import discover_split_dirs, validate_source_layout

if not raw_dir.is_dir():
    raise FileNotFoundError(
        f"Missing VisDrone raw directory: {raw_dir}\n"
        "Manually download/extract VisDrone2019-DET and place train/val/test-dev folders there."
    )

splits = validate_source_layout(raw_dir)
for name, info in splits.items():
    print(f"{name}: images={info.images_dir} annotations={info.annotations_dir}")

## 7. Run conversion if necessary

In [ ]:
from src.data.visdrone_convert import convert_visdrone

if RUN_CONVERSION:
    convert_visdrone(
        source=raw_dir,
        output=processed_dir,
        overwrite=OVERWRITE,
        limit=LIMIT,
    )
else:
    print("Skipping conversion (RUN_CONVERSION=False)")

## 8. Run dataset audit

In [ ]:
from pathlib import Path

from src.data.dataset_audit import audit_dataset, print_audit_summary, save_audit_report

report = audit_dataset(processed_dir)
report_path = Path(DRIVE_ROOT) / "outputs" / "metrics" / "visdrone_person_audit.json"
save_audit_report(report, report_path)
print_audit_summary(report)
print("Audit saved to:", report_path)

## 9. Display several converted samples

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import cv2

from src.data.visualize_annotations import visualize_split

vis_dir = Path(DRIVE_ROOT) / "outputs" / "visualizations" / "visdrone_person"
saved = visualize_split(processed_dir, "val", vis_dir, count=VIS_COUNT)

for path in saved[: min(4, len(saved))]:
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(path.name)
    plt.axis("off")
    plt.show()

## 10. Final dataset paths ready for YOLO training

In [ ]:
print("Processed dataset root:", processed_dir)
print("Train images:", processed_dir / "images" / "train")
print("Val images:", processed_dir / "images" / "val")
print("Test images:", processed_dir / "images" / "test")
print("Repo YOLO config template: configs/visdrone_person.yaml")
print("Next: notebooks/colab/02_train_person_detector.ipynb")